# Örgü ve Tekstil Kumaşlarında İlmek/Doku Kusurlarının Evrişimli Sinir Ağları (CNN) ile Tespiti
## Makine Öğrenmesi ile Probleme Uygun Çözüm Geliştirme Projesi

**Hazırlayan:** Proje Öğrencisi  
**Proje Türü:** Derin Öğrenme / Bilgisayarlı Görü (Computer Vision - CNN)  
**Hedef:** Endüstriyel örgü ve triko üretiminde kaçık ilmek, delik, yağ lekesi ve ilmek düğümü gibi doku kusurlarını görsel veri üzerinden otomatik olarak tespit etmek.

### 1. Kütüphanelerin Yüklenmesi ve Ortam Hazırlığı

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Tekrarlanabilirlik (Reproducibility) için sabit seed değerleri
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Sürümü: {tf.__version__}")
print(f"GPU Kullanılabilirliği: {tf.config.list_physical_devices('GPU')}")

### 2. Veri Setinin Oluşturulması / Yüklenmesi
Tekstil ve örgü dokuma literatüründe (AITEX / TILDA standartları) örgü kumaşlar ilmek sıraları (wales ve courses) boyunca periyodik sinüzoidal dalgalanmalardan oluşur.
Aşağıdaki fonksiyon, endüstriyel kalite kontrol kameralarından alınan makro örgü dokularını ve sık karşılaşılan 3 tip kusuru (kaçık ilmek/delik, yağ lekesi, ilmek düğümü) modellemektedir.

In [ ]:
def generate_knit_texture(size=128, is_defective=False, defect_type="none"):
    x = np.linspace(0, 16 * np.pi, size)
    y = np.linspace(0, 16 * np.pi, size)
    X, Y = np.meshgrid(x, y)
    
    wale_freq = 0.8
    course_freq = 0.8
    knit_base = (np.sin(X * wale_freq) * np.cos(Y * course_freq) + 
                 0.5 * np.sin(2 * X * wale_freq + np.pi/4) + 
                 0.3 * np.cos(2 * Y * course_freq))
    
    fiber_noise = np.random.normal(0, 0.25, (size, size))
    texture = knit_base + fiber_noise
    texture = (texture - texture.min()) / (texture.max() - texture.min() + 1e-6)
    texture = 0.2 + 0.6 * texture
    
    if is_defective:
        if defect_type == "none":
            defect_type = np.random.choice(["hole", "stain", "broken_stitch"])
        cx, cy = np.random.randint(30, size - 30, size=2)
        radius = np.random.randint(8, 18)
        dist_px = np.sqrt((np.arange(size)[:, None] - cy)**2 + (np.arange(size)[None, :] - cx)**2)
        
        if defect_type == "hole":
            mask = dist_px < radius
            texture[mask] = texture[mask] * 0.15 + np.random.normal(0.05, 0.02, np.sum(mask))
            edge_mask = (dist_px >= radius) & (dist_px < radius + 3)
            texture[edge_mask] += np.random.uniform(0.1, 0.3, np.sum(edge_mask))
        elif defect_type == "stain":
            smooth_drop = np.exp(- (dist_px**2) / (2 * (radius)**2))
            texture = texture * (1.0 - 0.65 * smooth_drop)
        elif defect_type == "broken_stitch":
            length = np.random.randint(20, 45)
            y_start = max(0, cy - 3)
            y_end = min(size, cy + 3)
            x_start = max(0, cx - length // 2)
            x_end = min(size, cx + length // 2)
            texture[y_start:y_end, x_start:x_end] = np.random.uniform(0.85, 0.98, (y_end - y_start, x_end - x_start))
            
    return np.clip(texture, 0.0, 1.0).astype(np.float32)

# 1000 örnekli veri seti üretimi (800 Normal, 200 Kusurlu -> Gerçekçi %80:%20 Sınıf Dengesizliği)
NUM_NORMAL = 800
NUM_DEFECTIVE = 200

images, labels, defect_types = [], [], []
for _ in range(NUM_NORMAL):
    images.append(generate_knit_texture(size=128, is_defective=False))
    labels.append(0)
    defect_types.append("Normal")

kinds = ["hole", "stain", "broken_stitch"]
for i in range(NUM_DEFECTIVE):
    k = kinds[i % len(kinds)]
    images.append(generate_knit_texture(size=128, is_defective=True, defect_type=k))
    labels.append(1)
    defect_types.append(k)

X = np.array(images)[..., np.newaxis]
y = np.array(labels, dtype=np.float32)
print(f"Veri Boyutu: {X.shape}, Etiket Boyutu: {y.shape}")

### 3. Keşifsel Veri Analizi (EDA)
Yönerge gereği en az 3 anlamlı görselleştirme hazırlanmıştır:
1. **Görsel 1:** Hatasız ve Kusurlu Örgü Numunelerinin Karşılaştırılması
2. **Görsel 2:** Sınıf Dağılımı ve Dengesizlik Analizi
3. **Görsel 3:** Piksel Yoğunluk Dağılımı (Histogram)

In [ ]:
# EDA 1: Örnek Görseller
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
sample_norm = np.where(y == 0)[0][:4]
sample_def = np.where(y == 1)[0][:4]

for i, idx in enumerate(sample_norm):
    axes[0, i].imshow(X[idx, :, :, 0], cmap='gray')
    axes[0, i].set_title(f"Hatasız Örgü ({i+1})", fontweight='bold', color='navy')
    axes[0, i].axis('off')

for i, idx in enumerate(sample_def):
    d_name = {"hole": "Delik/Kaçık", "stain": "Yağ Lekesi", "broken_stitch": "Düğüm"}.get(defect_types[idx])
    axes[1, i].imshow(X[idx, :, :, 0], cmap='gray')
    axes[1, i].set_title(f"Kusur: {d_name}", fontweight='bold', color='darkred')
    axes[1, i].axis('off')

plt.suptitle("Görsel 1: Hatasız ve Kusurlu Örgü Kumaş Örnekleri", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# EDA 2: Sınıf Dengesizliği Grafiği
plt.figure(figsize=(6, 4))
bars = plt.bar(["Hatasız (Normal)", "Kusurlu (Defective)"], [NUM_NORMAL, NUM_DEFECTIVE], color=['#2980b9', '#c0392b'], width=0.5)
plt.title("Görsel 2: Sınıf Dağılımı (Sınıf Dengesizliği)", fontweight='bold')
plt.ylabel("Gözlem Sayısı")
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 15, f"{int(h)} (%{h/10:.1f})", ha='center', fontweight='bold')
plt.ylim(0, 950)
plt.show()

In [ ]:
# EDA 3: Piksel Yoğunluk Dağılımı
plt.figure(figsize=(8, 4))
plt.hist(X[y == 0].flatten()[:80000], bins=50, alpha=0.6, label='Hatasız Örgü Pikselleri', color='#2980b9', density=True)
plt.hist(X[y == 1].flatten()[:80000], bins=50, alpha=0.6, label='Kusurlu Örgü Pikselleri', color='#e74c3c', density=True)
plt.title("Görsel 3: Piksel Yoğunluk Histogramı", fontweight='bold')
plt.xlabel("Normalize Piksel Değeri [0.0 - 1.0]")
plt.ylabel("Yoğunluk")
plt.legend()
plt.show()

### 4. Veri Ön İşleme, Veri Artırma ve Bölme
- **Veri Bölme:** %70 Eğitim, %15 Doğrulama, %15 Test (Stratified - sınıf oranları korunarak).
- **Veri Artırma (Data Augmentation):** Rastgele döndürme, yatay/dikey çevirme ve yakınlaştırma.
- **Sınıf Ağırlıkları (Class Weights):** Modelin azınlık sınıfı olan kusurları kaçırmaması için kayıp fonksiyonunda kusurlu sınıfa 4 kat ceza ağırlığı tanımlanması.

In [ ]:
idx_0 = np.where(y == 0)[0]
idx_1 = np.where(y == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

train_idx = np.concatenate([idx_0[:int(0.70*len(idx_0))], idx_1[:int(0.70*len(idx_1))]])
val_idx   = np.concatenate([idx_0[int(0.70*len(idx_0)):int(0.85*len(idx_0))], idx_1[int(0.70*len(idx_1)):int(0.85*len(idx_1))]])
test_idx  = np.concatenate([idx_0[int(0.85*len(idx_0)):], idx_1[int(0.85*len(idx_1)):]])

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]
X_test, y_test   = X[test_idx], y[test_idx]

class_weights = {
    0: (1 / (len(y_train) - np.sum(y_train))) * (len(y_train) / 2.0),
    1: (1 / np.sum(y_train)) * (len(y_train) / 2.0)
}
print(f"Eğitim: {len(X_train)}, Doğrulama: {len(X_val)}, Test: {len(X_test)}")
print(f"Sınıf Ağırlıkları: {class_weights}")

### 5. CNN Model Mimarisi Tasarımı

In [ ]:
model = models.Sequential([
    layers.Input(shape=(128, 128, 1)),
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    
    # 1. Evrişim Bloğu (Kenar ve temel ilmek hatları)
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # 2. Evrişim Bloğu (Doku örüntüsü ve periyodik yapı)
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # 3. Evrişim Bloğu (Kompleks kusur bölgeleri)
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Sınıflandırma Katmanı
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
], name="KnitwearDefectCNN")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)
model.summary()

### 6. Model Eğitimi

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=18,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

### 7. Model Değerlendirme ve Performans Grafikleri

In [ ]:
# Öğrenme Eğrileri
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history.history['loss'], label='Eğitim Kaybı', lw=2)
ax1.plot(history.history['val_loss'], label='Doğrulama Kaybı', lw=2, linestyle='--')
ax1.set_title('Kayıp (Loss) Eğrisi')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Eğitim Başarımı', lw=2)
ax2.plot(history.history['val_accuracy'], label='Doğrulama Başarımı', lw=2, linestyle='--')
ax2.set_title('Doğruluk (Accuracy) Eğrisi')
ax2.set_xlabel('Epoch')
ax2.legend()
plt.show()

In [ ]:
# Test Kümesi Metrikleri ve Karışıklık Matrisi
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)
y_true = y_test.astype(int)

TN = np.sum((y_true == 0) & (y_pred == 0))
FP = np.sum((y_true == 0) & (y_pred == 1))
FN = np.sum((y_true == 1) & (y_pred == 0))
TP = np.sum((y_true == 1) & (y_pred == 1))
cm = np.array([[TN, FP], [FN, TP]])

acc = (TP + TN) / (TP + TN + FP + FN)
prec = TP / (TP + FP + 1e-7)
rec = TP / (TP + FN + 1e-7)
f1 = 2 * (prec * rec) / (prec + rec + 1e-7)

print(f"Test Accuracy:  %{acc*100:.2f}")
print(f"Test Precision: %{prec*100:.2f}")
print(f"Test Recall:    %{rec*100:.2f}")
print(f"Test F1-Score:  {f1:.4f}")

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontweight='bold', fontsize=14, color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.xticks([0, 1], ['Tahmin: Normal', 'Tahmin: Kusurlu'])
plt.yticks([0, 1], ['Gerçek: Normal', 'Gerçek: Kusurlu'])
plt.title("Karışıklık Matrisi (Confusion Matrix)", fontweight='bold')
plt.colorbar()
plt.show()

### 8. Hata Analizi ve Yanlış Sınıflandırılan Örnekler
Endüstriyel açıdan False Negative (kusuru kaçırma) hatası, False Positive (hatasız ürüne kusurlu deme) hatasından çok daha kritiktir.

In [ ]:
fn_idx = np.where((y_true == 1) & (y_pred == 0))[0]
fp_idx = np.where((y_true == 0) & (y_pred == 1))[0]
print(f"Yanlış Negatif Sayısı (Kaçırılan Kusur): {len(fn_idx)}")
print(f"Yanlış Pozitif Sayısı (Yalancı Alarm):    {len(fp_idx)}")